# Web Map with Folium

**Folium** is a Python library for creating interactive web maps based on the JavaScript library Leaflet.js.

We have already used Folium many times — whenever we called the `.explore()` method on a `GeoDataFrame`. Under the hood, `GeoPandas` uses Folium to render those interactive maps.

Now we'll move to more flexible, hands-on map configuration directly through the Folium library.

In this section we will build an interactive map with the following layers:

- district boundary;
- land use;
- U-Bahn stations;
- walking isochrones (5, 10, and 15 minutes) from the stations.

## 0. Importing Libraries and Preparing Data

### 0.1. Importing Libraries

In [ ]:
import geopandas as gpd

import folium
from folium.plugins import MousePosition, Fullscreen, MiniMap

### 0.2. Preparing Data

In this example we use four datasets prepared in advance for Leopoldstadt — the second district of Vienna, an island between the Danube and the Danube Canal — in `data/leopoldstadt/`:

- `area.geojson` — district boundary;
- `landuse.geojson` — land use polygons;
- `metro.geojson` — U-Bahn stations;
- `isochrones.geojson` — walking isochrones from the stations.

_Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._

Reading them needs no ceremony by now — `read_file()` on each path, as in every module since the first.

In [ ]:
area = gpd.read_file("../../data/leopoldstadt/area.geojson")
landuse = gpd.read_file("../../data/leopoldstadt/landuse.geojson")
metro = gpd.read_file("../../data/leopoldstadt/metro.geojson")
isochrones = gpd.read_file("../../data/leopoldstadt/isochrones.geojson")

The boundary and the stations hold little more than a name apiece. The other two carry the fields the map will be built on, so those are worth a look before we start styling.

Land use is classified at three levels; we will colour the map by `LEV2`:

In [ ]:
landuse.head()

The isochrones carry the travel time in seconds (`value`), the station each was drawn from, and `population` — the number of residents the zone reaches, which we return to once the layer is on the map:

In [ ]:
isochrones.head()

Any of these layers can be thrown onto a quick map with `.explore()` at any point, as we have done throughout the course. Here we go straight to building the real one.

## 1. Creating the Base Map

Before adding thematic layers — the district boundary, land use, U-Bahn stations, and isochrones — we need to create the **map base**.

In this step we will:

1. determine the point where the map should open;
2. set the initial zoom level;
3. choose the background tile layer;
4. create the Folium map object that layers will be added to.

### 1.1. Setting the Map Centre

For the map to open centred on the study area, we calculate the geometric centroid of the district.

As we saw in the [second module](../module_2/projections_3.ipynb), geometric operations belong in a projected CRS, so we reproject the layer to its UTM zone, take the centroid there, and convert the result back to degrees — Folium expects latitude and longitude.

In [ ]:
centre = (
    area.to_crs(area.estimate_utm_crs())   # metric CRS
        .geometry.centroid                 # centroid of the district
        .to_crs(area.crs)                  # back to degrees
        .iloc[0]
)

centre_lat = centre.y
centre_lon = centre.x

print(f"Map centre: {centre_lat}, {centre_lon}")

### 1.2. Creating the Map Object

Now let's create the base Folium map:

- `folium.Map()` creates an interactive web map;
- `location` sets the map centre;
- `zoom_start` sets the initial zoom level — the larger the number, the closer the view; 13 frames a district like ours, while a whole city fits at 11–12;
- `tiles` selects the background tile layer;
- `control_scale` adds a scale bar to the map.

In [ ]:
m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=13,
    tiles="cartodbpositron",
    control_scale=True
)

m

## 2. Adding Layers

The base map is ready, but so far it contains only the background tile layer.

Now we'll gradually add spatial data as individual layers.

In Folium, data is typically added as `GeoJson` objects.

The general pattern looks like this:

```python
folium.GeoJson(
    data,
    display_parameters
).add_to(m)
```

### 2.1. District Boundary

We start with the most basic layer — the district boundary — to mark the study area.

#### 2.1.1. Defining the Style

The `style_function` is called separately for each GeoJSON feature. The `feature` object contains the geometry and attributes of that feature in `feature["properties"]`.

The style function must return a dictionary with the rendering parameters:

- `fillColor` — polygon fill colour;
- `color` — border colour;
- `weight` — line width;
- `fillOpacity` — fill opacity;
- `opacity` — border opacity;
- `dashArray` — makes the line dashed.

In [ ]:
def area_style(feature):
    return {
        "fillColor": "#64748B",
        "color": "#334155",
        "weight": 2,
        "fillOpacity": 0.03,
        "opacity": 0.65,
        "dashArray": "6, 5",
    }

#### 2.1.2. Adding the GeoJSON Layer

Add the layer to the map.

In [ ]:
folium.GeoJson(
    area,
    name="District Boundary",
    style_function=area_style,
).add_to(m)

m

### 2.2. Land Use

Now we move on to a more complex thematic layer — land use. Unlike the previous examples, this dataset contains multiple categories of features with different land use types. Before styling, we'll first look at which categories exist in the dataset, and if needed, merge some of them into broader groups.

This is the one layer here that does not come from OpenStreetMap. Vienna surveys the actual use of every parcel in the city and publishes the result as the **Realnutzungskartierung**, which covers the district wall to wall — no gaps — and classifies each polygon at three levels of detail: `LEV1` is the coarsest (three classes), `LEV2` has eleven, and `LEV3` goes down to individual uses such as a swimming pool or a railway yard. OpenStreetMap does have a `landuse` tag, but in Vienna it is mapped only patchily: it covers about two thirds of this district, and seven polygons in eight are a single lawn or flowerbed.

We will map `LEV2` — detailed enough to be interesting, coarse enough to fit in a legend.

#### 2.2.1. Exploring Categories

First, which land use types are present in the data.

In [ ]:
landuse["LEV2"].value_counts()

#### 2.2.2. Grouping Categories

Eleven categories is more than a reader can hold at once, so we'll merge related ones into seven broader groups. We create a mapping dictionary from the original land use types to the new groups.

The keys are the German category names exactly as they appear in the data. Leaving them untranslated is deliberate: these are the values the file actually contains, and a dictionary that quietly renames them would stop matching the moment you re-download the layer. The translation belongs in the legend, which we build further down.

In [ ]:
landuse_groups = {
    "Wohn- u. Mischnutzung (Schwerpunkt Wohnen)": "residential",

    "Geschäfts,- Kern- und Mischnutzung (Schwerpunkt betriebl. Tätigkeit)": "business",
    "Industrie- und Gewerbenutzung": "business",

    "soziale Infrastruktur": "social",

    "Erholungs- u. Freizeiteinrichtungen": "recreation",

    "Naturraum": "nature",
    "Landwirtschaft": "nature",

    "Gewässer": "water",

    "Straßenraum": "transport",
    "weitere verkehrliche Nutzungen": "transport",
    "Technische Infrastruktur/Kunstbauten/Sondernutzung": "transport",
}

#### 2.2.3. Colors for Each Group

Assign a distinct colour to each group.

Two of the seven are not free choices: water is blue and nature is green, because a map that breaks those conventions is read wrong before it is read at all. Transport takes a neutral grey — it is the background against which the rest is read, not a category anyone is looking for. The remaining four take distinct hues.

One caveat worth stating plainly: seven categorical fills is past the point where colour alone can be told apart reliably, particularly for readers with colour vision deficiency. That is why the legend below is not optional and the tooltip names the category — the colour narrows it down, the label settles it.

In [ ]:
landuse_colors = {
    "residential": "#eda100",
    "business": "#eb6834",
    "social": "#e87ba4",
    "recreation": "#1baf7a",
    "nature": "#008300",
    "water": "#2a78d6",
    "transport": "#9aa0a6",
}

#### 2.2.4. Style Function

Now let's write a function that automatically assigns a style to each feature.

`landuse_style(feature)` is called separately for each GeoJSON feature.
The `feature` argument is the current layer feature including its geometry and attributes, stored in `feature["properties"]`.

Inside the function:

1. The `LEV2` field value is extracted from the feature's attributes.
2. The `landuse_groups` dictionary maps the raw land use type to its broader group.
3. The group's colour is looked up in `landuse_colors`.
4. The function returns a dictionary with the rendering style for that feature.

Both lookups have a fallback: a land use type that is missing from `landuse_groups` falls into `"other"` and is drawn in neutral grey. All eleven categories the city uses are mapped, so nothing should reach the fallback — but it keeps the map working if a later survey adds one.

In [ ]:
def landuse_style(feature):

    landuse_type = feature["properties"].get("LEV2")

    landuse_class = landuse_groups.get(landuse_type, "other")

    color = landuse_colors.get(landuse_class, "#d9d9d9")

    return {
        "fillColor": color,
        "color": "#ffffff",
        "weight": 0.4,
        "fillOpacity": 0.5,
        "opacity": 0.6,
    }

#### 2.2.5. Configuring Tooltips

Tooltips appear when the user hovers over a feature.

The tooltip will display two fields: `LEV2`, the category we coloured the map by, and `LEV3`, the finer classification underneath it — so hovering over a green polygon tells the reader whether it is a park, a meadow or a wood.
`aliases` sets the field labels, and `localize=True` ensures values are formatted correctly.

In [ ]:
landuse_tooltip = folium.GeoJsonTooltip(
    fields=["LEV2", "LEV3"],
    aliases=["Land Use Type:", "Detail:"],
    localize=True
)

#### 2.2.6. Adding the Layer to the Map

Now we add the land use layer to the map using `folium.GeoJson`, passing the data, the style function, and the tooltips.

`style_function=landuse_style` means Folium will call the style function for each GeoJSON feature, automatically determining its colour and rendering parameters based on its attributes.

In [ ]:
folium.GeoJson(
    landuse,
    name="Land Use",
    style_function=landuse_style,
    tooltip=landuse_tooltip
).add_to(m)

m

### 2.3. Walking Isochrones

Isochrones delineate the area reachable within a given travel time. In our example we use walking isochrones for three time thresholds: 5, 10, and 15 minutes.

#### 2.3.1. Defining Styles for Each Zone

First, let's create a dictionary of rendering styles for each isochrone.
The keys are travel times in seconds:

- `300` — 5 minutes;
- `600` — 10 minutes;
- `900` — 15 minutes.

For each zone we set the line rendering parameters:

- `color` — line colour;
- `weight` — line width;
- `opacity` — opacity.

In [ ]:
isochrone_styles = {
    300: {
        "color": "#475569",
        "weight": 3.5,
        "opacity": 0.95,
    },
    600: {
        "color": "#64748B",
        "weight": 3,
        "opacity": 0.85,
    },
    900: {
        "color": "#94A3B8",
        "weight": 2.5,
        "opacity": 0.75,
    },
}


#### 2.3.2. Style Function

Now let's write a function that automatically assigns a style to each isochrone.

As in the previous examples, the function receives a `feature` object — a single GeoJSON feature with its attributes. Here, the attributes store the travel time value.

The travel time in seconds is extracted from `feature["properties"]["value"]`:

- `300` — 5 minutes;
- `600` — 10 minutes;
- `900` — 15 minutes.

The rendering parameters for that zone are then looked up in `isochrone_styles`. If the value is not in the dictionary, a default style is used.

The function returns a dictionary of line rendering parameters for the isochrone.

In [ ]:
def isochrone_style(feature):
    value = int(feature["properties"].get("value", 0))
    style = isochrone_styles.get(value, {
        "color": "#9CA3AF",
        "weight": 2,
        "opacity": 0.7,
    })

    return {
        "fill": False,
        "color": style["color"],
        "weight": style["weight"],
        "opacity": style["opacity"],
    }

#### 2.3.3. Merging the Zones

The file holds one isochrone per station per threshold — nine stations times three thresholds, 27 polygons in all. Drawn as they come, they pile on top of one another: at the 15-minute threshold more than half of the area covered is drawn twice or more, and the map becomes a tangle of outlines that no reader can follow.

The question a reader actually brings to this map is not "how far is it from this particular station" but "how far is it from the U-Bahn". So we merge the isochrones of all stations into a single shape per threshold, using `dissolve()` — the same operation as in the [third module](../module_3/geoprocessing_1.ipynb): group by travel time, then union the geometries within each group.

That leaves three nested zones in place of 27 overlapping ones. There is a cost: once nine stations are merged into one shape, no single station can be named in the tooltip, so it will show the travel time alone. If you need the per-station detail, keep the original layer as a second, switchable layer.

In [ ]:
# One shape per travel time: dissolve unions the isochrones of all stations
isochrones_layer = (
    isochrones[["value", "population", "geometry"]]
    .dissolve(by="value")          # aggfunc="first" carries population through
    .reset_index()
)

isochrones_layer["value"] = isochrones_layer["value"].astype(int)
isochrones_layer["minutes"] = (isochrones_layer["value"] / 60).astype(int)

print(f"{len(isochrones)} isochrones from {isochrones['station_name'].nunique()} stations"
      f" -> {len(isochrones_layer)} merged zones")

#### 2.3.4. Configuring Tooltips

The isochrones get a tooltip of their own, showing the travel time in minutes and `population` — the number of residents the zone reaches, which is the figure the whole project is aimed at.

`aliases` sets the field label, and `sticky=True` keeps the tooltip pinned near the cursor.

In [ ]:
isochrones_tooltip = folium.GeoJsonTooltip(
    fields=["minutes", "population"],
    aliases=["Walking minutes:", "Residents reached:"],
    localize=True,
    sticky=True
)

#### 2.3.5. Adding the Layer to the Map

Add the isochrone layer to the map using `folium.GeoJson`.

`style_function=isochrone_style` tells Folium to call `isochrone_style` for each feature, selecting the line colour, width, and opacity based on the travel time value.

In [ ]:
folium.GeoJson(
    isochrones_layer,
    name="Isochrones",
    style_function=isochrone_style,
    tooltip=isochrones_tooltip
).add_to(m)

m

#### 2.3.6. What the Zones Actually Say

The isochrones carry one more field, and it is the one the whole project has been working towards. `population` holds the number of people living inside each zone, taken from the WorldPop raster of the [fifth module](../module_5/rasters_2.ipynb). The zones were clipped to the district before counting — they spill well past its boundary, and uncut they would credit Leopoldstadt with other districts' residents.

Set the three figures against the ground they cover and the point of the exercise appears:

| within | residents | share of the district's people | share of its area |
| --- | --- | --- | --- |
| 5 minutes | 23,280 | 26 % | 13 % |
| 10 minutes | 55,858 | 62 % | 35 % |
| 15 minutes | 67,256 | **74 %** | **51 %** |

A fifteen-minute walk from a U-Bahn station reaches **three quarters of the people who live in Leopoldstadt while covering only half of its ground**. The half left out is the Prater and the banks of the Danube: parkland and water, where nobody lives.

That gap is the whole argument for measuring accessibility in people rather than in square kilometres — and it is what the zonal statistics of the fifth module were for. In the assignment, your own map carries the same figures for each category of amenity.

### 2.4. U-Bahn Stations

The last layer — U-Bahn stations.

Unlike the previous examples, here we work with point features, which are displayed using markers rather than styled shapes.

We will draw each station as the badge Vienna itself uses: a white **U** on a square in the colour of its line. The `line` column carries the line number, joined onto the stations from the city's own register.

#### 2.4.1. Line Colours

The five U-Bahn lines have official colours, set by Wiener Linien and used on every map and sign in the city. Reusing them means a reader who knows Vienna reads the map without the legend at all.

Only three of the five run through Leopoldstadt, but we map all five, so the same dictionary works for any district.

In [ ]:
line_colors = {
    1: "#E20A17",   # U1 — red
    2: "#A862A4",   # U2 — purple
    3: "#EF7C00",   # U3 — orange
    4: "#00A650",   # U4 — green
    6: "#A6673A",   # U6 — brown
}

#### 2.4.2. Building the Badge

A coloured square with a letter in it is not one of Folium's built-in markers, so we build it out of HTML with **`folium.DivIcon`** — a marker whose content is an arbitrary block of HTML, styled with CSS.

Two details in the CSS are doing real work rather than decoration. The **white border** and the **soft shadow** separate the badge from whatever is beneath it: without them the green U4 badge would blend into the green of the nature polygons, and the red U1 badge would sit uneasily on the orange of the business ones. And `icon_anchor` is set to half the icon size, which centres the badge on the station instead of hanging it below and to the right — the default, and a common source of markers that look subtly misplaced.

In [ ]:
def station_badge(line):
    color = line_colors.get(line, "#334155")

    return folium.DivIcon(
        icon_size=(22, 22),
        icon_anchor=(11, 11),      # half the size: centres the badge on the point
        html=f'''
            <div style="
                width: 22px;
                height: 22px;
                background: {color};
                border: 2px solid #ffffff;
                border-radius: 5px;
                box-shadow: 0 1px 3px rgba(0, 0, 0, 0.4);
                color: #ffffff;
                font: 700 13px/18px sans-serif;
                text-align: center;
            ">U</div>
        ''',
    )

#### 2.4.3. Adding the Layer to the Map

The previous layers went on with a single `folium.GeoJson` call, but that will not work here. `GeoJson` takes **one** marker object and reuses it for every feature, so every station would come out the same colour. To give each its own badge we add the markers one at a time, in a loop.

They still need to behave as one layer in the control panel, so we collect them into a **`folium.FeatureGroup`** — a container that gathers any number of objects under a single name and a single switch. Here it holds markers, but it will take anything you can add to a map: points, lines and polygons together, which is how you would group a category of amenities with its catchment areas.

The tooltip is built in the same loop. There is no `GeoJsonTooltip` here, since we are no longer going through `GeoJson`: a plain string on each marker does the job.

One caveat about the data. Praterstern and Schottenring are interchanges served by two lines each, but the register records a single line per station, so their badges show only one of the two.

In [ ]:
stations = folium.FeatureGroup(name="U-Bahn Stations")

for _, station in metro.iterrows():
    folium.Marker(
        location=[station.geometry.y, station.geometry.x],
        icon=station_badge(station["line"]),
        tooltip=f"{station['name']} — U{station['line']}",
    ).add_to(stations)

stations.add_to(m)

m

## 3. Map Controls

After adding all thematic layers, let's configure the map controls.

### 3.1. Layer Control

Add a layer control panel that lets users toggle individual layers on and off.

In [ ]:
folium.LayerControl(collapsed=False).add_to(m)

`LayerControl` lets users toggle layers on and off. `collapsed=False` keeps the panel expanded by default.

### 3.2. Cursor Coordinates

Add a cursor coordinate display. When the user moves the mouse over the map, the current latitude and longitude are shown.

In [ ]:
MousePosition().add_to(m)

### 3.3. Fullscreen Button

Add a button that expands the map to the full browser window — useful when the map is embedded in a page alongside other content.

Parameters:

- `position="bottomright"` places the button in the bottom-right corner;
- `title` sets the tooltip text when entering fullscreen;
- `title_cancel` sets the tooltip text when exiting fullscreen;
- `force_separate_button=True` renders the button as a standalone UI element.

In [ ]:
Fullscreen(
    position="bottomright",
    title="Enter fullscreen",
    title_cancel="Exit fullscreen",
    force_separate_button=True,
).add_to(m)

### 3.4. Mini Map

The mini map helps users understand where the current map view sits relative to a broader area.

In [ ]:
MiniMap(tile_layer="cartodbpositron", toggle_display=True).add_to(m)

### 3.5. Legend

A map with seven land use colours, three isochrone line weights and a badge per U-Bahn line is unreadable without a legend — the reader has no way to tell what a colour means.

Folium has no built-in legend for `GeoJson` layers. The `branca` colour maps that Folium ships with draw a **continuous gradient bar**, which suits a numeric scale (population, density, year of construction) but misrepresents categories: land use groups have no order, and a gradient implies one. So for a categorical layer the usual approach is to add a small block of HTML on top of the map.

Two decisions are worth spelling out:

1. **We generate the legend from the same dictionaries that drive the styling** — `landuse_colors` and `isochrone_styles`. This is the whole point: a legend written by hand drifts out of sync as soon as someone changes a colour, and a map whose legend lies is worse than a map with no legend at all. Here, changing a colour in `landuse_colors` changes both the map and its legend.
2. **The legend is attached to the map root, not added as a layer** — via `m.get_root().html.add_child(...)`. It is page furniture rather than spatial data, so it should not appear in the layer control panel and should not disappear when a layer is switched off.

The only thing we still need is a set of human-readable labels. This is where the German category names get translated — and where the short group keys (`nature`, `transport`), fine in code, turn into something a reader can use.

In [ ]:
landuse_labels = {
    "residential": "Housing and mixed use",
    "business": "Business and industry",
    "social": "Social infrastructure",
    "recreation": "Recreation and leisure",
    "nature": "Nature and farmland",
    "water": "Water",
    "transport": "Transport and utilities",
}

Now we assemble the legend itself. The three loops build one row per land use group, one per isochrone zone and one per U-Bahn line; every swatch, line and badge is painted from the same dictionary the map itself draws from, so none of them can drift out of step.

The block is styled with a little CSS: `position: fixed` pins it to the corner of the map frame, `z-index` keeps it above the tiles, and the bottom offset leaves room for the scale bar.

One detail to keep in mind: on the map the polygons are drawn with `fillOpacity` 0.5, so the basemap shows through and the colours look paler there than in the legend swatches. Full-strength swatches stay legible at 14 pixels, which is why they are drawn that way here.

In [ ]:
# Land use: one row per group, the swatch colour taken from landuse_colors
landuse_rows = "".join(
    f"""
    <div class="legend-row">
        <span class="legend-swatch" style="background: {landuse_colors[group]}"></span>
        {label}
    </div>"""
    for group, label in landuse_labels.items()
)

# U-Bahn: one row per line actually present on the map
line_rows = "".join(
    f"""
    <div class="legend-row">
        <span class="legend-badge" style="background: {line_colors[number]}">U</span>
        U{number}
    </div>"""
    for number in sorted(metro["line"].unique())
)

# Isochrones: one row per zone, the line styled exactly as on the map
isochrone_rows = "".join(
    f"""
    <div class="legend-row">
        <span class="legend-line" style="border-top: {style["weight"]}px solid {style["color"]}"></span>
        {seconds // 60} min walk
    </div>"""
    for seconds, style in isochrone_styles.items()
)

legend_html = f"""
<div class="map-legend">
    <div class="legend-title">Land use</div>
    {landuse_rows}
    <div class="legend-title">Walking isochrones</div>
    {isochrone_rows}
    <div class="legend-title">U-Bahn lines</div>
    {line_rows}
</div>

<style>
.map-legend {{
    position: fixed;
    bottom: 42px;   /* leaves room for the scale bar */
    left: 12px;
    z-index: 9999;
    background: rgba(255, 255, 255, 0.92);
    padding: 10px 14px;
    border: 1px solid #cbd5e1;
    border-radius: 6px;
    font-family: sans-serif;
    font-size: 12px;
    line-height: 1.6;
    color: #1e293b;
}}
.legend-title {{ font-weight: 600; margin: 4px 0 2px; }}
.legend-row {{ display: flex; align-items: center; gap: 8px; }}
.legend-swatch {{
    width: 14px;
    height: 14px;
    border: 1px solid #ffffff;
    border-radius: 2px;
    display: inline-block;
}}
.legend-line {{ width: 18px; display: inline-block; }}
.legend-badge {{
    width: 14px;
    height: 14px;
    border-radius: 3px;
    color: #ffffff;
    font: 700 9px/14px sans-serif;
    text-align: center;
    display: inline-block;
}}
</style>
"""

m.get_root().html.add_child(folium.Element(legend_html))

m

## 4. Viewing and Saving the Map

The map is now complete.

We have added:

- thematic layers;
- feature styling;
- tooltips;
- map controls.

Time to look at the finished map and save it as an HTML file.

### 4.1. Final Map

Display the finished interactive map.

In [ ]:
m

### 4.2. Saving the Map

Save the map to an HTML file — it can then be opened in any browser. The path is relative to the notebook, so the file lands next to it. The line is left commented out here; uncomment it when you want the file, and see the [next section](map_2.ipynb) for publishing it online.

In [ ]:
# m.save("index.html")

## Summary

In this section we built an interactive web map with multiple layers.

We:

- loaded and inspected the data;
- created a base map;
- added thematic layers;
- configured styles and tooltips;
- added map controls;
- looked at how to save the map as HTML — publishing it is the subject of the [next section](map_2.ipynb).